# DATA CLEANING(PART 1)

In [2]:
import pandas as pd
import numpy as np

In [3]:
df=pd.read_csv("../../datasets/titanic/train.csv")
# df.head()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
# from sklearn.preprocessing import LabelEncoder
# le=LabelEncoder()
# df["Sex"]=le.fit_transform(df["Sex"])
# df.head()

In [4]:
from sklearn.preprocessing import OneHotEncoder
ohe=OneHotEncoder(sparse_output=False, drop="first",dtype=int)
sex_encoded=ohe.fit_transform(df[["Sex"]])
encoded_df = pd.DataFrame(
    sex_encoded, columns=ohe.get_feature_names_out(["Sex"])
)
df= pd.concat([df,encoded_df],axis=1)
# df.head()

In [ ]:
df.head()

In [5]:
df=df.drop(columns=["Sex"])
# df.head()

In [6]:
df["Age"]=df["Age"].fillna(df["Age"].median())
df["Embarked"]=df["Embarked"].fillna(df["Embarked"].mode()[0])

In [7]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         0
Sex_male         0
dtype: int64

In [8]:
df=pd.get_dummies(df,drop_first=True,columns=["Embarked"],dtype=float)
# df.head()

In [9]:
df.head(3)

,PassengerId,Survived,Pclass,Name,Age,SibSp,Parch,Ticket,Fare,Cabin,Sex_male,Embarked_Q,Embarked_S
0,1,0,3,"Braund, Mr. Owen Harris",22.0,1,0,A/5 21171,7.2500,NaN,1,0.0,1.0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,1,0,PC 17599,71.2833,C85,0,0.0,0.0
2,3,1,3,"Heikkinen, Miss. Laina",26.0,0,0,STON/O2. 3101282,7.9250,NaN,0,0.0,1.0


In [10]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
df[["Age","Fare"]]=scaler.fit_transform(df[["Age","Fare"]])
df.head()

,PassengerId,Survived,Pclass,Name,Age,SibSp,Parch,Ticket,Fare,Cabin,Sex_male,Embarked_Q,Embarked_S
0,1,0,3,"Braund, Mr. Owen Harris",-0.565736,1,0,A/5 21171,-0.502445,NaN,1,0.0,1.0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",0.663861,1,0,PC 17599,0.786845,C85,0,0.0,0.0
2,3,1,3,"Heikkinen, Miss. Laina",-0.258337,0,0,STON/O2. 3101282,-0.488854,NaN,0,0.0,1.0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",0.433312,1,0,113803,0.420730,C123,0,0.0,1.0
4,5,0,3,"Allen, Mr. William Henry",0.433312,0,0,373450,-0.486337,NaN,1,0.0,1.0


In [ ]:
df.shape

## XGBOOST - EXTREME GRADIENT BOOSTING

In [13]:
drop_columns=["PassengerId","Name","Ticket","Cabin"]
df1=df.copy()
df=df.drop(columns=drop_columns)
df1.head(1)

KeyError: "['PassengerId', 'Name', 'Ticket', 'Cabin'] not found in axis"

In [14]:
df.head(2)

,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S
0,0,3,-0.565736,1,0,-0.502445,1,0.0,1.0
1,1,1,0.663861,1,0,0.786845,0,0.0,0.0


In [15]:
y=df["Survived"]
X=df.drop(columns=["Survived"])
X.head()

,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S
0,3,-0.565736,1,0,-0.502445,1,0.0,1.0
1,1,0.663861,1,0,0.786845,0,0.0,0.0
2,3,-0.258337,0,0,-0.488854,0,0.0,1.0
3,1,0.433312,1,0,0.420730,0,0.0,1.0
4,3,0.433312,0,0,-0.486337,1,0.0,1.0


In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2, random_state=42)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier 
from xgboost import XGBClassifier

In [ ]:
tree=DecisionTreeClassifier(random_state=42)
tree.fit(X_train,y_train)

In [ ]:
forest=RandomForestClassifier(
    n_estimators=100, random_state=42
)
forest.fit(X_train,y_train)

In [ ]:
print(X.dtypes)
print(X.columns.tolist())
print(y.dtypes)

In [ ]:
boost=XGBClassifier(
    eval_metric="logloss",
    random_state=42
)
boost.fit(X_train,y_train)

In [ ]:
models = {
    "Decision Tree": tree,
    "Random Forest": forest,
    "XGBoost": boost
}
for key,value in models.items():
    y_pred=value.predict(X_test)
    acc=accuracy_score(y_test,y_pred)
    print(f"{key} : {acc}")

In [ ]:
import pandas as pd

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": boost.feature_importances_
})

print(
    importance.sort_values(
        by="Importance",
        ascending=False
    )
)

In [ ]:
df['Name']=df1["Name"]
df.head()

In [ ]:
# Family size
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

# Is passenger alone?
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

# Title from Name
df["Title"] = df["Name"].str.extract(' ([A-Za-z]+)\.', expand=False)

In [16]:
df.head(3)

,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S
0,0,3,-0.565736,1,0,-0.502445,1,0.0,1.0
1,1,1,0.663861,1,0,0.786845,0,0.0,0.0
2,1,3,-0.258337,0,0,-0.488854,0,0.0,1.0


## Grid Search CV - K-fold CV

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression